In [36]:
# New project to build an agentic AI application that read and writes information to a report-style document.
import os
from typing_extensions import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages, BaseMessage
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_community.tools import ReadFileTool, WriteFileTool
from langchain_core.messages import AIMessage

In [37]:
# Setting up variables from the .env file and initializing the model:
load_dotenv(".env")
MODEL_NAME = os.getenv("MODEL_NAME")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
model = init_chat_model(MODEL_NAME, api_key=OPENAI_API_KEY, temperature=0.5, stream_usage = True)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_community.tools import WriteFileTool
import nbformat
import uuid

# Defining the notebook analyzer tool function:
@tool
def analyze_notebook(file_path: str) -> str:
    """Safely read and summarize files."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
        
        lines = [f"Notebook: {file_path} | Cells: {len(nb.cells)}"]
        for i, cell in enumerate(nb.cells[:40]):
            if cell.cell_type == "markdown":
                text = cell.source.strip()[:600]
                if text:
                    lines.append(f"\n[Markdown] {text}")
            elif cell.cell_type == "code":
                code = cell.source.strip()[:500]
                if code:
                    lines.append(f"\n[Code] {code}")
                    if cell.outputs:
                        lines.append("  [output truncated]")
        return "\n".join(lines)
    except Exception as e:
        return f"Error: {str(e)}"


# Defining the file creator tool function
@tool
def file_creator_node(file_path: str, text: str) -> str:
    """Save content to a file. Use parameter name 'text' (required by WriteFileTool)."""
    try:
        writer = WriteFileTool(root_dir="./reports")
        result = writer.invoke({
            "file_path": file_path,
            "text": text
        })
        return f"✅ Successfully saved: ./reports/{file_path}"
    except Exception as e:
        return f"❌ Save error: {str(e)}"


# Defining the agent:
tools = [analyze_notebook, file_creator_node]
checkpointer = MemorySaver()

system_message = SystemMessage(content="""You are an expert data science assistant.
- Always start by calling analyze_notebook on .ipynb files.
- Generate clear, structured, professional reports with the goal of persuading non-technical stakeholders and C suite executives.
- Always use file_creator_node to save the final report.""")

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
)


# Executing the tool:
def run_query(query: str):
    thread_id = str(uuid.uuid4())
    
    result = agent.invoke(
        {"messages": [system_message, HumanMessage(content=query)]},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    final_msg = result["messages"][-1].content
    print("\n" + "="*75)
    print("FINAL ANSWER")
    print("="*75)
    print(final_msg)


# Defining the trigger by leveraging user input:
if __name__ == "__main__":
    while True:
        user_input = input(f"Enter a file name stored in the parent directory of this file: ")
        if user_input == 'quit'.lower():
            break
        run_query(user_input)


FINAL ANSWER
Created the notebook review report and saved it here:

`./reports/Social_Media_Impact_on_Mental_Health_Report.md`
